# 1. Cristallographie géométrique - Structure du matériau


In [9]:
import numpy as np
import matplotlib.pyplot as plt

from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.diffraction.xrd import XRDCalculator

from pymatgen.electronic_structure.plotter import BSPlotter
from pymatgen.phonon.plotter import PhononBSPlotter

from scipy.optimize import curve_fit
from scipy.constants import hbar, k as k_B

API_KEY = "vKJsFu0jdhLy7CJj5Mwar6S68kxgMc3n"
MP_ID = "mp-1008556"  

with MPRester(API_KEY) as mpr:
    structure = mpr.get_structure_by_material_id(MP_ID)


## 1.1 Réseau direct
À produire :
- commentaire : maille primitive / conventionnelle / nombre d'atomes = 4



In [10]:
print("Formule :", structure.composition.reduced_formula)

lattice = structure.lattice
a_vec, b_vec, c_vec = lattice.matrix
print("Vecteurs de base du réseau direct: ")
print("a =", a_vec)
print("b =", b_vec)
print("c =", c_vec)
print("Les normes (a,b,c) sont ", structure.lattice.abc, "et les angles [°C] (alpha,beta,gamma)sont",structure.lattice.angles)
print("Volume Ω =", lattice.volume, "Å³")

Formule : AlGaN2
Vecteurs de base du réseau direct: 
a = [-2.22093546 -2.22093546 -0.        ]
b = [-2.22093546  2.22093546  0.        ]
c = [-0.         -0.         -4.42383794]
Les normes (a,b,c) sont  (3.140877048687329, 3.140877048687329, 4.42383794) et les angles [°C] (alpha,beta,gamma)sont (90.0, 90.0, 90.0)
Volume Ω = 43.64164186160555 Å³


Le réseau est défini par 3 vecteurs orthogonaux où les paramètres de maille sont a=b≠c et les vecteurs de maille sont: **a** = a $\hat{x}$, **b** = a $\hat{y}$, **c** = c $\hat{z}$. Tout point du réseau peut s'écrire $\mathbf{R}= l \times \mathbf{a} + m \times \mathbf{b} + n \times \mathbf{c} $  
Le volume Ω se calcule comme suit:  **Ω** = **a**.(**b**x**c**) = **b**.(**c**x**a**) = **c**.(**a**x**b**) = **a**²**c** = 43.64 Å³ 

## 1.2 Réseau réciproque

In [12]:
rec = lattice.reciprocal_lattice
b1, b2, b3 = rec.matrix
print("a* =", b1)
print("b* =", b2)
print("c* =", b3)

a* = [-1.41453577 -1.41453577 -0.        ]
b* = [-1.41453577  1.41453577 -0.        ]
c* = [-0.          0.         -1.42030187]


Les vecteurs $a*$, $b*$, $c*$ sont les vecteurs du réseau réciproque formés à partir des 3 vecteurs du réseau direct.   
On peut les trouver par:  
- $a*$= 2π $\frac{(\mathbf{b} \times \mathbf{c})}{Ω}$= $\frac{2π}{\mathbf{a}}$
- $b*$= 2π $\frac{(\mathbf{c} \times \mathbf{a})}{Ω}$= $\frac{2π}{\mathbf{b}}$
- $c*$= 2π $\frac{(\mathbf{a} \times \mathbf{b})}{Ω}$= $\frac{2π}{\mathbf{c}}$

Comme pour le réseau direct, les 3 vecteurs sont orthogonaux. 
Le réseau réciproque est défini tel que : $ \mathbf{K} =h \times \mathbf{a*}+ k \times \mathbf{b*}+ l \times \mathbf{c*}$.  
Nous avons $e^{i \times \mathbf{K} \times \mathbf{R}}=0 $.   

Le réseau réciproque est utile pour pouvoir trouver les zones de Brillouin


## 1.3 Système cristallin, type de maille et groupe ponctuel

In [14]:
sga = SpacegroupAnalyzer(structure)
spg_symbol = sga.get_space_group_symbol()
spg_number = sga.get_space_group_number()
crys_system = sga.get_crystal_system()
point_group = sga.get_point_group_symbol()

print("Système cristallin:", crys_system)
print("Groupe spatial:", spg_symbol, "(#", spg_number, ")")
print("Groupe ponctuel:", point_group)

Système cristallin: tetragonal
Groupe spatial: P-4m2 (# 115 )
Groupe ponctuel: -42m


Le système cristallin est tétragonal car les paramètres de mailles vérifient a=b≠c avec des angles (alpha,beta,gamma)=(90°,90°,90°).   

Le type de maille est Primitive (noté P dans le groupe d'espace).  
Le groupe spatial est obtenu en combinant le groupe ponctuel et le type de maille.   
Le numéro 115 représente le numéro international.   

Pour le groupe ponctuel, le 1er symbole reflète la symétrie le long de l'axe **c** (c'est à dire $\bar{4}$ --> opération de roto-inversion).   
Ensuite, le deuxième symbole donne la direction selon les directions [100] et [010] (2--> axe de rotation d'ordre 2).  
Enfin, le dernier symbole montre la symétrie le long des directions [110] et [1 $\bar{1}$ 0] (ici m signifie un système avec un plan miroir). 